# Audio Understanding with Qwen2-Audio

This notebook demonstrates how to use Qwen/Qwen2-Audio-7B-Instruct for audio understanding tasks. Qwen2-Audio is a multimodal model that can process audio inputs and answer questions about them, provide descriptions, and engage in multi-turn conversations about audio content.

Key capabilities:
- Audio Question Answering
- Audio Captioning and Description
- Multi-turn conversations about audio content
- Support for various audio formats and tasks

In [ ]:
!pip install transformers torch librosa accelerate

## Load Model and Processor

In [ ]:
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor
import torch

# Load the model and processor
model_name = "Qwen/Qwen2-Audio-7B-Instruct"

processor = AutoProcessor.from_pretrained(model_name)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"Model loaded successfully on device: {model.device}")

## Prepare Audio Input

In [ ]:
import librosa
import requests
import numpy as np
from io import BytesIO

# Load a sample audio file from a URL
# Using a sample audio from Hugging Face datasets
audio_url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac"

# Download and load the audio
response = requests.get(audio_url)
audio_bytes = BytesIO(response.content)

# Load and resample to 16kHz
audio_array, sampling_rate = librosa.load(audio_bytes, sr=16000)

print(f"Audio loaded successfully")
print(f"Sampling rate: {sampling_rate} Hz")
print(f"Duration: {len(audio_array) / sampling_rate:.2f} seconds")
print(f"Audio shape: {audio_array.shape}")

## Audio Question Answering

In [ ]:
# Create a conversation with audio and text prompt
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio_url": "placeholder"},
            {"type": "text", "text": "What is being said in this audio?"}
        ]
    }
]

# Process the inputs
text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
inputs = processor(
    text=text,
    audios=[audio_array],
    sampling_rate=sampling_rate,
    return_tensors="pt",
    padding=True
)
inputs = inputs.to(model.device)

# Generate response
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False
    )

# Decode the response
response = processor.batch_decode(
    output_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)[0]

print("Question: What is being said in this audio?")
print(f"Answer: {response}")

## Audio Captioning

In [ ]:
# Ask the model to describe/caption the audio content
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio_url": "placeholder"},
            {"type": "text", "text": "Describe this audio in detail. What sounds do you hear?"}
        ]
    }
]

# Process the inputs
text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
inputs = processor(
    text=text,
    audios=[audio_array],
    sampling_rate=sampling_rate,
    return_tensors="pt",
    padding=True
)
inputs = inputs.to(model.device)

# Generate caption
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False
    )

# Decode the caption
caption = processor.batch_decode(
    output_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)[0]

print("Audio Caption:")
print(caption)

## Multi-turn Conversation

In [ ]:
# Multi-turn conversation about the same audio
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio_url": "placeholder"},
            {"type": "text", "text": "What type of audio is this?"}
        ]
    }
]

# First turn
text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
inputs = processor(
    text=text,
    audios=[audio_array],
    sampling_rate=sampling_rate,
    return_tensors="pt",
    padding=True
)
inputs = inputs.to(model.device)

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)

first_response = processor.batch_decode(
    output_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)[0]

print("Turn 1:")
print(f"User: What type of audio is this?")
print(f"Assistant: {first_response}")
print()

# Add assistant response and follow-up question
conversation.append({"role": "assistant", "content": first_response})
conversation.append({
    "role": "user",
    "content": [{"type": "text", "text": "Can you identify any background noise or other details?"}]
})

# Second turn
text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
inputs = processor(
    text=text,
    audios=[audio_array],
    sampling_rate=sampling_rate,
    return_tensors="pt",
    padding=True
)
inputs = inputs.to(model.device)

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)

second_response = processor.batch_decode(
    output_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)[0]

print("Turn 2:")
print(f"User: Can you identify any background noise or other details?")
print(f"Assistant: {second_response}")